# Fine-tune del detector de cabezas — dataset filtrado (≥ 5 cabezas/frame)

**Hipótesis:** ~65% de los frames de `s03` (la fuente dominante, 70% del dataset) están mal etiquetados
(mediana **1** cabeza marcada en escenas de multitud). Eso enseña al modelo que "cabeza visible = fondo"
y produce el subconteo crónico (golden bias **-1.87**, recall **0.79**).

Este notebook construye `bus_head_v5_min5` = solo frames con **≥ 5 cabezas etiquetadas**, y entrena un
fine-tune **desde el modelo base** `models/yolov5mu-head-base.pt` (nunca desde el best.pt de una ronda
previa — receta validada R2–R5).

> **Cómo corre:** el host NO tiene ultralytics/torch. El entrenamiento se ejecuta dentro de la imagen
> `mot-dev:latest` vía Docker en el engine del host (`docker -c default`, con `--runtime=nvidia`).
> Docker Desktop NO pasa CUDA. Las celdas de construcción del dataset solo usan la stdlib, así que
> cualquier kernel de Python sirve.

**Comparar contra el campeón R5:** golden count-MAE **2.04**, recall **0.792**, mAP50 **0.872**.

## 0. Configuración

In [ ]:
import os, sys, shutil, subprocess, hashlib, re
from pathlib import Path

# --- rutas en el HOST ---
REPO        = Path('/home/camilo-pc/Multi-Object-Tracking')
SRC         = REPO / 'data' / 'bus_head_v5'          # dataset origen (completo)
DST_NAME    = 'bus_head_v5_min5'                       # dataset filtrado a crear
DST         = REPO / 'data' / DST_NAME
YAML_PATH   = REPO / 'data' / f'{DST_NAME}.yaml'
MIN_HEADS   = 5                                       # umbral: descartar frames con < 5 cajas

# --- entrenamiento (dentro de mot-dev:latest, rutas relativas a /workspace) ---
BASE_MODEL  = 'models/yolov5mu-head-base.pt'
DATA_YAML   = f'data/{DST_NAME}.yaml'
PROJECT     = 'outputs/head_detector'
RUN_NAME    = 'yolo-bus-head-min5'
EPOCHS, IMGSZ, BATCH, WORKERS = 80, 640, 8, 4
CONTAINER   = 'train-min5'
DOCKER_IMG  = 'mot-dev:latest'

def docker(args, **kw):
    """Ejecuta docker en el engine del host (default). Devuelve CompletedProcess."""
    cmd = ['docker', '-c', 'default'] + args
    return subprocess.run(cmd, text=True, capture_output=True, **kw)

print('REPO  :', REPO)
print('SRC   :', SRC, '(existe:', SRC.exists(), ')')
print('DST   :', DST)
print('umbral:', MIN_HEADS, 'cabezas/frame')

## 1. Construir el dataset filtrado (≥ 5 cabezas/frame)

Copia a `bus_head_v5_min5/` solo los frames cuyo label tiene **≥ 5 cajas**. Imagen + label se copian
juntos. Reporta cuántos quedan por split y por fuente de vídeo.

In [ ]:
def n_boxes(label_path: Path) -> int:
    """Número de cajas = líneas no vacías en el .txt de YOLO."""
    if not label_path.exists():
        return 0
    with open(label_path) as f:
        return sum(1 for ln in f if ln.strip())

def source_of(stem: str) -> str:
    """Agrupa por fuente quitando el sufijo _fNNNN del nombre del frame."""
    s = re.sub(r'_f?\d+$', '', stem)
    return re.sub(r'\d+$', '', s) or stem

# limpiar destino previo para que el experimento sea reproducible
if DST.exists():
    shutil.rmtree(DST)

stats = {}            # split -> (kept, dropped)
by_source = {}        # source -> kept
total_boxes = 0

for split in ('train', 'val'):
    img_dir = SRC / 'images' / split
    lbl_dir = SRC / 'labels' / split
    out_img = DST / 'images' / split
    out_lbl = DST / 'labels' / split
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    kept = dropped = 0
    for img in sorted(img_dir.iterdir()):
        if img.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        lbl = lbl_dir / (img.stem + '.txt')
        nb = n_boxes(lbl)
        if nb >= MIN_HEADS:
            shutil.copy2(img, out_img / img.name)
            shutil.copy2(lbl, out_lbl / lbl.name)
            kept += 1
            total_boxes += nb
            src = source_of(img.stem)
            by_source[src] = by_source.get(src, 0) + 1
        else:
            dropped += 1
    stats[split] = (kept, dropped)
    print(f'[{split}] copiados={kept}  descartados={dropped}  (de {kept+dropped})')

tot_keep = sum(k for k, _ in stats.values())
tot_all  = sum(k + d for k, d in stats.values())
print(f'\nTOTAL: {tot_all} -> {tot_keep} frames  ({tot_all - tot_keep} descartados, '
      f'{100*(tot_all-tot_keep)/tot_all:.0f}%)')
print(f'cajas totales conservadas: {total_boxes}')
print('\nFrames conservados por fuente:')
for src, n in sorted(by_source.items(), key=lambda x: -x[1]):
    print(f'  {src:12s} {n}')

## 2. Anti-fuga del golden

El golden (151 frames, congelado) NO puede aparecer en el entrenamiento. `bus_head_v5` ya pasó esta
verificación, y este dataset es un subconjunto suyo — pero confirmamos de todos modos comparando
**cámara + número de frame**.

In [ ]:
GOLDEN = REPO / 'data' / 'golden' / 'images' / 'val'

def cam_frame(stem: str):
    """Extrae (fuente, n_frame) para comparar identidad de frame entre datasets."""
    m = re.search(r'(\d+)$', stem.replace('_f', '_'))
    num = m.group(1) if m else ''
    return (source_of(stem), num)

golden_ids = {cam_frame(p.stem) for p in GOLDEN.iterdir()
              if p.suffix.lower() in ('.jpg', '.jpeg', '.png')} if GOLDEN.exists() else set()

train_ids = set()
for split in ('train', 'val'):
    for p in (DST / 'images' / split).iterdir():
        train_ids.add(cam_frame(p.stem))

leak = golden_ids & train_ids
print(f'golden ids: {len(golden_ids)} | dataset ids: {len(train_ids)}')
if leak:
    print('\n⚠️  POSIBLE FUGA — revisar antes de entrenar:')
    for x in sorted(leak):
        print('   ', x)
    raise SystemExit('Fuga del golden detectada — ABORTAR')
else:
    print('\n✅ sin fuga del golden')

## 3. Escribir el YAML del dataset

In [ ]:
yaml_text = (
    f'path: /workspace/data/{DST_NAME}\n'
    'train: images/train\n'
    'val: images/val\n'
    '\n'
    'names:\n'
    '  0: head\n'
)
YAML_PATH.write_text(yaml_text)
print('escrito:', YAML_PATH)
print(yaml_text)

## 4. Pre-check de GPU (engine del host)

Debe imprimir la RTX 4060 Ti. Si falla, NO entrenes (Docker Desktop no pasa CUDA — usa siempre `docker -c default`).

In [ ]:
r = docker(['run', '--rm', '--runtime=nvidia', '-e', 'NVIDIA_VISIBLE_DEVICES=all',
            DOCKER_IMG, 'nvidia-smi',
            '--query-gpu=name,memory.total,memory.used', '--format=csv,noheader'])
print(r.stdout.strip() or r.stderr.strip())
assert r.returncode == 0, 'GPU no disponible en el engine del host'

## 5. Lanzar el entrenamiento (en background, ~1–2 h)

Se lanza **detached** (`-d`) para que sobreviva aunque se desconecte el kernel. Gotchas horneados:
`--shm-size=8g` (si no, los workers del DataLoader crashean), `batch=8 workers=4 cache=False` +
`--memory 16g` (el auto-batch congeló el PC una vez).

In [ ]:
# limpiar contenedor previo con el mismo nombre
docker(['rm', '-f', CONTAINER])

train_cmd = (
    f'yolo detect train model={BASE_MODEL} data={DATA_YAML} '
    f'epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} workers={WORKERS} cache=False device=0 '
    f'project={PROJECT} name={RUN_NAME}'
)

r = docker(['run', '-d', '--name', CONTAINER, '--runtime=nvidia',
            '-e', 'NVIDIA_VISIBLE_DEVICES=all',
            '--memory', '16g', '--memory-swap', '16g', '--shm-size=8g',
            '-v', f'{REPO}:/workspace', '-w', '/workspace',
            DOCKER_IMG] + train_cmd.split())

if r.returncode == 0:
    print('✅ entrenamiento lanzado. Container:', r.stdout.strip()[:12])
    print('   comando:', train_cmd)
else:
    print('❌ fallo al lanzar:\n', r.stderr)

## 6. Monitorear el progreso

Re-ejecuta esta celda para refrescar (muestra estado + últimas líneas del log). No bloquea el kernel.

In [ ]:
status = docker(['ps', '-a', '--filter', f'name={CONTAINER}',
                 '--format', '{{.Status}}']).stdout.strip()
print('estado:', status or '(no existe)')
print('-' * 70)
logs = docker(['logs', '--tail', '30', CONTAINER])
print((logs.stdout + logs.stderr).strip()[-4000:])

## 7. Localizar `best.pt`, sha256 y limpiar

Ejecutar **cuando el entrenamiento termine** (estado `Exited (0)`). Gotcha: con `project=` relativo,
los pesos a veces quedan anidados bajo `runs/detect/` — se buscan y se reubican.

In [ ]:
expected = REPO / PROJECT / RUN_NAME / 'weights' / 'best.pt'
cands = sorted(REPO.rglob('best.pt'),
               key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)
cands = [p for p in cands if RUN_NAME in str(p)]
print('candidatos best.pt:')
for p in cands:
    print('  ', p)

best = cands[0] if cands else None
if best and best != expected:
    expected.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best, expected)
    print('\nreubicado a:', expected)
    best = expected

if best and best.exists():
    sha = hashlib.sha256(best.read_bytes()).hexdigest()
    print('\nbest.pt :', best)
    print('sha256  :', sha)
else:
    print('\n⚠️  best.pt aún no existe — ¿ya terminó el entrenamiento?')

# limpiar el contenedor de entrenamiento
docker(['rm', '-f', CONTAINER])

## 8. Evaluar contra el golden

Quick check de recall/mAP sobre el golden (151 frames). Para el **veredicto oficial count-MAE vs R5**
(la métrica que decide desplegar o no), usa el skill `/eval-golden`, que compara contra el campeón y
actualiza `docs/golden_baseline.md`.

In [ ]:
val_cmd = (
    f'yolo val model={PROJECT}/{RUN_NAME}/weights/best.pt '
    f'data=data/golden/golden.yaml imgsz={IMGSZ} conf=0.25 iou=0.5 '
    f'project={PROJECT} name={RUN_NAME}-golden'
)
r = docker(['run', '--rm', '--runtime=nvidia', '-e', 'NVIDIA_VISIBLE_DEVICES=all',
            '--shm-size=8g', '-v', f'{REPO}:/workspace', '-w', '/workspace',
            DOCKER_IMG] + val_cmd.split())
out = (r.stdout + r.stderr)
# imprime las líneas de métricas (P, R, mAP50, mAP50-95)
for ln in out.splitlines():
    if any(k in ln for k in ('Class', 'all', 'head', 'Images', 'Speed')):
        print(ln)
print('\nCampeón R5 a batir — recall 0.792 | mAP50 0.872 | count-MAE 2.04')
print('Para el count-MAE oficial vs R5:  ejecuta el skill  /eval-golden')